In [ ]:
!pip install meteocalc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold
from tqdm import tqdm_notebook as tqdm
import datetime
from meteocalc import feels_like, Temp
from sklearn import metrics
import gc
import os
# Adjust input paths if necessary when running locally
KAGGLE_INPUT_PATH = '/kaggle/input' # or './input' if data is local
if not os.path.exists(KAGGLE_INPUT_PATH):
    KAGGLE_INPUT_PATH = '.' # Adjust if your local data is elsewhere
    # Create dummy files if they don't exist for local run, or ensure they are present
    os.makedirs(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction'), exist_ok=True)
    os.makedirs(os.path.join(KAGGLE_INPUT_PATH, 'rows-to-drop'), exist_ok=True)
    # Example: !touch ./ashrae-energy-prediction/train.csv ... etc.

for dirname, _, filenames in os.walk(KAGGLE_INPUT_PATH):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Load Data

In [ ]:
# Original code from https://www.kaggle.com/gemartin/load-data-reduce-memory-usage by @gemartin

from pandas.api.types import is_datetime64_any_dtype as is_datetime
from pandas.api.types import is_categorical_dtype

def reduce_mem_usage(df, use_float16=False):
    """
    Iterate through all the columns of a dataframe and modify the data type to reduce memory usage.        
    """
    
    start_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage of dataframe is {:.2f} MB".format(start_mem))
    
    for col in df.columns:
        if is_datetime(df[col]) or is_categorical_dtype(df[col]):
            continue
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if use_float16 and c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype("category")

    end_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage after optimization is: {:.2f} MB".format(end_mem))
    print("Decreased by {:.1f}%".format(100 * (start_mem - end_mem) / start_mem))
    
    return df

In [ ]:
train_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/train.csv'))
building_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/building_metadata.csv'))
weather_train_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/weather_train.csv')) # Renamed to avoid conflict

Referring to the following discussions. We have filtered the data in the next step

* Ref1. https://www.kaggle.com/c/ashrae-energy-prediction/discussion/114830#latest-680086
* Ref2. https://www.kaggle.com/c/ashrae-energy-prediction/discussion/113054#656588 

In [ ]:
# Ref 1
#train_df = train_df [ train_df['building_id'] != 1099 ]
# Ref 2
#train_df = train_df.query('not (building_id <= 104 & meter == 0 & timestamp <= "2016-05-20")')

In [ ]:
# eliminate bad rows
bad_rows_path = os.path.join(KAGGLE_INPUT_PATH, 'rows-to-drop/rows_to_drop.csv')
if os.path.exists(bad_rows_path):
    bad_rows = pd.read_csv(bad_rows_path)
    train_df.drop(bad_rows.loc[:, '0'], inplace = True)
    train_df.reset_index(drop = True, inplace = True)
else:
    print(f"Warning: bad_rows file not found at {bad_rows_path}. Skipping this step.")

In [ ]:
# Original code from https://www.kaggle.com/aitude/ashrae-missing-weather-data-handling by @aitude
def fill_weather_dataset(weather_df):
    
    # Find Missing Dates
    time_format = "%Y-%m-%d %H:%M:%S"
    start_date = datetime.datetime.strptime(weather_df['timestamp'].min(),time_format)
    end_date = datetime.datetime.strptime(weather_df['timestamp'].max(),time_format)
    total_hours = int(((end_date - start_date).total_seconds() + 3600) / 3600)
    hours_list = [(end_date - datetime.timedelta(hours=x)).strftime(time_format) for x in range(total_hours)]

    missing_hours = []
    for site_id in range(16):
        site_hours = np.array(weather_df[weather_df['site_id'] == site_id]['timestamp'])
        new_rows = pd.DataFrame(np.setdiff1d(hours_list,site_hours),columns=['timestamp'])
        new_rows['site_id'] = site_id
        weather_df = pd.concat([weather_df,new_rows])

        weather_df = weather_df.reset_index(drop=True)           

    # Add new Features
    weather_df["datetime"] = pd.to_datetime(weather_df["timestamp"])
    weather_df["day"] = weather_df["datetime"].dt.day
    weather_df["week"] = weather_df["datetime"].dt.isocalendar().week.astype(int) # updated for pandas >= 1.1.0
    weather_df["month"] = weather_df["datetime"].dt.month
    
    # Reset Index for Fast Update
    weather_df = weather_df.set_index(['site_id','day','month'])

    air_temperature_filler = pd.DataFrame(weather_df.groupby(['site_id','day','month'])['air_temperature'].mean(),columns=["air_temperature"])
    weather_df.update(air_temperature_filler,overwrite=False)

    # Step 1
    cloud_coverage_filler = weather_df.groupby(['site_id','day','month'])['cloud_coverage'].mean()
    # Step 2
    cloud_coverage_filler = pd.DataFrame(cloud_coverage_filler.fillna(method='ffill'),columns=["cloud_coverage"])

    weather_df.update(cloud_coverage_filler,overwrite=False)

    due_temperature_filler = pd.DataFrame(weather_df.groupby(['site_id','day','month'])['dew_temperature'].mean(),columns=["dew_temperature"])
    weather_df.update(due_temperature_filler,overwrite=False)

    # Step 1
    sea_level_filler = weather_df.groupby(['site_id','day','month'])['sea_level_pressure'].mean()
    # Step 2
    sea_level_filler = pd.DataFrame(sea_level_filler.fillna(method='ffill'),columns=['sea_level_pressure'])

    weather_df.update(sea_level_filler,overwrite=False)

    wind_direction_filler =  pd.DataFrame(weather_df.groupby(['site_id','day','month'])['wind_direction'].mean(),columns=['wind_direction'])
    weather_df.update(wind_direction_filler,overwrite=False)

    wind_speed_filler =  pd.DataFrame(weather_df.groupby(['site_id','day','month'])['wind_speed'].mean(),columns=['wind_speed'])
    weather_df.update(wind_speed_filler,overwrite=False)

    # Step 1
    precip_depth_filler = weather_df.groupby(['site_id','day','month'])['precip_depth_1_hr'].mean()
    # Step 2
    precip_depth_filler = pd.DataFrame(precip_depth_filler.fillna(method='ffill'),columns=['precip_depth_1_hr'])

    weather_df.update(precip_depth_filler,overwrite=False)

    weather_df = weather_df.reset_index()
    weather_df = weather_df.drop(['datetime','day','week','month'],axis=1)
    
    def get_meteorological_features(data):
        def calculate_rh(df):
            df['relative_humidity'] = 100 * (np.exp((17.625 * df['dew_temperature']) / (243.04 + df['dew_temperature'])) / np.exp((17.625 * df['air_temperature'])/(243.04 + df['air_temperature'])))
        def calculate_fl(df):
            flike_final = []
            flike = []
            # calculate Feels Like temperature
            for i in range(len(df)):
                at = df['air_temperature'].iloc[i] # Use .iloc for safety
                rh = df['relative_humidity'].iloc[i]
                ws = df['wind_speed'].iloc[i]
                # meteocalc might expect Temp object, ensure compatibility or handle potential NaN/None
                if pd.notnull(at) and pd.notnull(rh) and pd.notnull(ws):
                    flike.append(feels_like(Temp(at, unit = 'C'), rh, ws))
                else:
                    flike.append(Temp(np.nan, unit = 'C')) # Append NaN if inputs are invalid

            for i in range(len(flike)):
                flike_final.append(flike[i].f)
            df['feels_like'] = flike_final
            del flike_final, flike
        calculate_rh(data)
        calculate_fl(data)
        return data

    weather_df = get_meteorological_features(weather_df)
    return weather_df

In [ ]:
def features_engineering(df):
    
    # Sort by timestamp
    df.sort_values("timestamp")
    df.reset_index(drop=True)
    
    # Add more features
    df["timestamp"] = pd.to_datetime(df["timestamp"],format="%Y-%m-%d %H:%M:%S")
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    
    df['month_group'] = df['timestamp'].dt.month # Renamed from 'month' to avoid clash with weather_df month
    df['month_group'].replace((1, 2, 3, 4), 1, inplace = True)
    df['month_group'].replace((5, 6, 7, 8), 2, inplace = True)
    df['month_group'].replace((9, 10, 11, 12), 3, inplace = True)
  
    df['square_feet'] =  np.log1p(df['square_feet'])
    
    # Remove Unused Columns
    drop = ["timestamp"]
    df = df.drop(drop, axis=1)
    gc.collect()
    
    # Encode Categorical Data
    le = LabelEncoder()
    df["primary_use"] = le.fit_transform(df["primary_use"])
    
    return df

In [ ]:
# weather manipulation for train data
weather_train_df = fill_weather_dataset(weather_train_df)

# memory reduction
train_df = reduce_mem_usage(train_df,use_float16=True)
building_df = reduce_mem_usage(building_df,use_float16=True)
weather_train_df = reduce_mem_usage(weather_train_df,use_float16=True)

# merge data for training
train_df = train_df.merge(building_df, left_on='building_id',right_on='building_id',how='left')
# Merge with weather_train_df, ensuring 'timestamp' is in the correct format if not already handled by features_engineering
train_df['timestamp_merge_key'] = pd.to_datetime(train_df["timestamp"],format="%Y-%m-%d %H:%M:%S") if 'timestamp' in train_df else None
weather_train_df['timestamp_merge_key'] = pd.to_datetime(weather_train_df["timestamp"],format="%Y-%m-%d %H:%M:%S") if 'timestamp' in weather_train_df else None
train_df = train_df.merge(weather_train_df,how='left',on=['site_id','timestamp_merge_key'],suffixes=('', '_weather'))
train_df.drop('timestamp_merge_key', axis=1, inplace=True, errors='ignore')
if 'timestamp_weather' in train_df.columns: train_df.drop('timestamp_weather', axis=1, inplace=True) # drop redundant timestamp from weather
if 'timestamp' not in train_df.columns and 'timestamp_x' in train_df.columns: # handle potential suffixes from merge
    train_df.rename(columns={'timestamp_x':'timestamp'}, inplace=True)

del weather_train_df # Delete after merge
gc.collect()

In [ ]:
# feature engineering for training data
train_df = features_engineering(train_df)

# transform target variable
train_df['meter_reading'] = np.log1p(train_df["meter_reading"])

In [ ]:
# declare target, categorical and numeric columns
target = 'meter_reading'
categorical = ['building_id', 'site_id', 'primary_use', 'meter','dayofweek']
# Ensure 'month_group' used for stratification is in categorical if it's not already part of 'features'
# features will be defined dynamically based on processed columns
numeric_cols = [col for col in train_df.columns if col not in categorical + [target, 'timestamp', 'month_group'] and train_df[col].dtype != 'object']
features = categorical + numeric_cols

In [ ]:
import seaborn as sns

def run_lgbm(train, cat_features = None, num_rounds = 20000, folds = 5):
    # Ensure cat_features is correctly passed or defined, using global 'categorical' if None
    if cat_features is None:
        cat_features = categorical # Assuming 'categorical' is defined globally
    
    # Use 'month_group' for stratification, ensure it exists in train_df
    # If 'month_group' is not in train_df, this will error. It's created in features_engineering.
    kf = StratifiedKFold(n_splits=folds, shuffle=False, random_state=2319) # Removed random_state for non-reproducible folds if data changes
    models = []
    feature_importance_df = pd.DataFrame()

    param =  {'num_leaves': 3160,
             'objective': 'regression',
             'learning_rate': 0.03,
             'boosting': 'gbdt',
             'subsample': 0.5,
             'feature_fraction': 0.7,
             'n_jobs': -1,
             'seed': 50,
             'metric': 'rmse'
              }
    
    oof = np.zeros(len(train))
  
    # Check if 'month_group' exists for stratification
    if 'month_group' not in train.columns:
        raise ValueError("'month_group' not found in training data for stratification. Please ensure features_engineering is run.")

    for fold_idx, (tr_idx, val_idx) in enumerate(tqdm(kf.split(train, train['month_group']), total = folds)):
        tr_x, tr_y = train[features].iloc[tr_idx], train[target].iloc[tr_idx]
        vl_x, vl_y = train[features].iloc[val_idx], train[target].iloc[val_idx]
        tr_data = lgb.Dataset(tr_x, label = tr_y,  categorical_feature = cat_features)
        vl_data = lgb.Dataset(vl_x, label = vl_y,  categorical_feature = cat_features)
        clf = lgb.train(param, tr_data, num_rounds, valid_sets = [tr_data, vl_data], verbose_eval = 25, 
                        early_stopping_rounds = 50)
        
        clf.save_model('lgbm_model_fold_{}.txt'.format(fold_idx)) # Save model
        
        fold_importance_df = pd.DataFrame()
        fold_importance_df["feature"] = features
        fold_importance_df["importance"] = clf.feature_importance()
        
        feature_importance_df = pd.concat([feature_importance_df, fold_importance_df], axis=0)
        models.append(clf)
        oof[val_idx] = clf.predict(vl_x)
        gc.collect()
    score = np.sqrt(metrics.mean_squared_error(train[target], np.clip(oof, a_min=0, a_max=None)))
    print('Our oof cv is :', score)
    
    cols = (feature_importance_df[["feature", "importance"]]
        .groupby("feature")
        .mean()
        .sort_values(by="importance", ascending=False)[:20].index)
    best_features = feature_importance_df.loc[feature_importance_df.feature.isin(cols)]

    plt.figure(figsize=(14,26))
    sns.barplot(x="importance", y="feature", data=best_features.sort_values(by="importance",ascending=False))
    plt.title('LightGBM Features (averaged over folds)')
    plt.tight_layout()
    # Save plot if needed, ensure directory exists if saving to a specific path
    # plt.savefig('lgbm_importances.png') 

    return models # models are returned but also saved to disk

# This cell trains and saves the models. 
# It might take a long time to run.
# Set a flag to control execution of model training for pipeline runs
TRAIN_MODELS = True # Set to True to train and save models, False to skip and use saved ones

if TRAIN_MODELS:
    print("Training and saving models...")
    models_trained = run_lgbm(train_df, cat_features=categorical) # Pass categorical features explicitly
    print("Models trained and saved.")
else:
    print("Skipping model training. Ensure models are already saved if you intend to predict.")

In [ ]:
# read test
test_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/test.csv'))
row_ids = test_df["row_id"]
test_df.drop("row_id", axis=1, inplace=True)
test_df = reduce_mem_usage(test_df)

# Load building_df again if it was deleted or for a clean prediction path
# Or ensure building_df from training part is available
building_df_pred = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/building_metadata.csv'))
building_df_pred = reduce_mem_usage(building_df_pred, use_float16=True)

test_df = test_df.merge(building_df_pred,left_on='building_id',right_on='building_id',how='left')
del building_df_pred # Delete after merge
gc.collect()

# fill test weather data
weather_test_df = pd.read_csv(os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/weather_test.csv'))
weather_test_df = fill_weather_dataset(weather_test_df)
weather_test_df = reduce_mem_usage(weather_test_df)

# merge weather data
# Ensure 'timestamp' is in the correct format for merging
test_df['timestamp_merge_key'] = pd.to_datetime(test_df["timestamp"],format="%Y-%m-%d %H:%M:%S") if 'timestamp' in test_df else None
weather_test_df['timestamp_merge_key'] = pd.to_datetime(weather_test_df["timestamp"],format="%Y-%m-%d %H:%M:%S") if 'timestamp' in weather_test_df else None
test_df = test_df.merge(weather_test_df,how='left',on=['site_id','timestamp_merge_key'], suffixes=('', '_weather'))
test_df.drop('timestamp_merge_key', axis=1, inplace=True, errors='ignore')
if 'timestamp_weather' in test_df.columns: test_df.drop('timestamp_weather', axis=1, inplace=True)
if 'timestamp' not in test_df.columns and 'timestamp_x' in test_df.columns: 
    test_df.rename(columns={'timestamp_x':'timestamp'}, inplace=True)

del weather_test_df # Delete after merge
gc.collect()

# feature engineering for test data
test_df = features_engineering(test_df)
# Ensure 'features' list is defined based on columns available in test_df after processing
# This should match the 'features' used during training
global features # Make sure 'features' from training setup is accessible or redefine if necessary
expected_features = features # From training cell. Check consistency.
test_features = [f for f in expected_features if f in test_df.columns]
if len(test_features) != len(expected_features):
    print("Warning: Feature mismatch between train and test. Ensure consistent processing.")
    # Potentially add missing columns with default values (e.g., 0 or mean/median from training)
    # For now, we'll proceed with common features, but this might impact model performance.
    # features = test_features # Or handle more robustly

In [ ]:
def generate_predictions_from_saved_models(test_df, num_folds=5, iterations = 120):
    # Load models
    models = []
    print(f"Loading {num_folds} models...")
    for fold_idx in range(num_folds):
        model_path = f'lgbm_model_fold_{fold_idx}.txt'
        if not os.path.exists(model_path):
            print(f"Model file {model_path} not found. Ensure models are trained and saved or TRAIN_MODELS is True.")
            # Depending on desired behavior, could raise error or skip prediction
            return # Or raise Exception("Model file not found")
        model = lgb.Booster(model_file=model_path) 
        models.append(model)
    print("Models loaded.")
    
    # Ensure 'features' is defined and available. It should be the same list as used in training.
    # This might need to be explicitly passed or ensured it's in the global scope from training setup.
    # For this pipeline, 'features' list defined in cell 10 should be accessible.
    global features 
    
    # split test data into batches
    set_size = len(test_df)
    if set_size == 0:
        print("Test data is empty. Skipping predictions.")
        return
    batch_size = (set_size + iterations - 1) // iterations # Ensure all data is covered
    meter_reading = []
    print(f"Starting predictions for {set_size} rows in {iterations} iterations (batch size ~{batch_size})...")
    for i in tqdm(range(iterations)):
        pos = i*batch_size
        end_pos = min(pos + batch_size, set_size)
        if pos >= end_pos:
            continue # Should not happen with correct batch_size calculation
        
        current_batch_df = test_df[features].iloc[pos : end_pos]
        fold_preds = [np.expm1(model.predict(current_batch_df)) for model in models]
        meter_reading.extend(np.mean(fold_preds, axis=0))

    print(f"Total predictions generated: {len(meter_reading)}")
    assert len(meter_reading) == set_size, f"Prediction length mismatch: {len(meter_reading)} vs {set_size}"
    
    sample_submission_path = os.path.join(KAGGLE_INPUT_PATH, 'ashrae-energy-prediction/sample_submission.csv')
    if os.path.exists(sample_submission_path):
        submission = pd.read_csv(sample_submission_path)
        # Ensure row_ids used for submission matches the test_df from which predictions were made
        # The original row_ids were stored at the beginning of cell 12. Assuming it's available.
        global row_ids
        if len(row_ids) != len(meter_reading):
             print(f"Warning: row_ids length ({len(row_ids)}) and meter_reading length ({len(meter_reading)}) mismatch.")
             # This might happen if test_df was filtered or changed post row_id extraction. For now, we proceed.
             # A robust solution might re-index or ensure row_ids directly correspond to the final test_df.
             # For this exercise, we'll assume row_ids from original test_df is fine if length matches.
             # If submission['row_id'] is to be used, it must match the prediction order.
        submission_df = pd.DataFrame({'row_id': row_ids[:len(meter_reading)], 'meter_reading': meter_reading})
        submission_df['meter_reading'] = np.clip(submission_df['meter_reading'], a_min=0, a_max=None) # clip min at zero
        submission_df.to_csv('submission.csv', index=False)
        print('Submission file created: submission.csv')
    else:
        print(f"Sample submission file not found at {sample_submission_path}. Cannot create submission file.")
    print('We are done with predictions!')

# Call the prediction function. 
# This will load models if TRAIN_MODELS was False, or use freshly trained ones if True (though run_lgbm returns them, this func reloads from disk)
generate_predictions_from_saved_models(test_df, num_folds=5) # Assuming 5 folds were used in training